# 05 — Results and Figures for Paper

Generate all tables and figures used in the IEEE paper.
This notebook consolidates results from notebooks 02–04 and produces
publication-ready plots saved to `paper/figures/`.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

Path('../paper/figures').mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 150, 'savefig.bbox': 'tight'})

## Table I — Cross-lingual Transfer Results (Paper 1)

In [ ]:
table1 = pd.DataFrame({
    'Model':       ['AfriBERTa', 'BantuBERTa', 'mBERT', 'BiGRU', 'TextCNN', 'CharCNN'],
    'Type':        ['Transformer']*3 + ['Traditional']*3,
    'Acc (before)': [74.21, 74.54, 58.72, 24.04, 21.90, 19.16],
    'F1 (before)':  [74.74, 73.75, 59.17, 23.00, 23.20, 16.21],
    'Acc (after)':  [88.30, 86.57, 84.62, 83.32, 59.13, 48.79],
    'F1 (after)':   [87.87, 86.06, 84.22, 87.90, 57.32, 47.64],
})
print('Table I: Cross-lingual Transfer Results')
print(table1.to_string(index=False))

## Table II — Catastrophic Forgetting Scores

In [ ]:
table2 = pd.DataFrame({
    'Model':           ['AfriBERTa', 'mBERT', 'BiGRU', 'TextCNN'],
    'Src Acc Before':  [84.20,       82.30,   88.51,   90.23],
    'Src Acc After':   [79.89,       79.81,   23.31,   22.68],
    'Forgetting %':    [5.14,        3.03,    73.68,   74.86],
})
print('Table II: Catastrophic Forgetting')
print(table2.to_string(index=False))

## Figure 1 — Model Accuracy Comparison Bar Chart

In [ ]:
models = ['AfriBERTa', 'BantuBERTa', 'mBERT', 'BiGRU', 'TextCNN', 'CharCNN']
before = [74.21, 74.54, 58.72, 24.04, 21.90, 19.16]
after  = [88.30, 86.57, 84.62, 83.32, 59.13, 48.79]

x = np.arange(len(models))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, before, w, label='Zero-shot (before FT)', color='#5B9BD5')
ax.bar(x + w/2, after,  w, label='Fine-tuned (after FT)',  color='#ED7D31')
ax.set_xticks(x)
ax.set_xticklabels(models, rotation=15)
ax.set_ylabel('Accuracy on Kirundi (%)')
ax.set_title('Cross-lingual Transfer: Zero-shot vs Fine-tuned')
ax.axhline(y=80, color='gray', linestyle='--', alpha=0.5)
ax.legend()
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig('../paper/figures/crosslingual_accuracy.pdf')
plt.show()

## Table III — MAGE Results by Language (Paper 2)

In [ ]:
table3 = pd.DataFrame({
    'Language':   ['Kinyarwanda', 'Swahili', 'Xitsonga'],
    'Baseline LR':  [54.20, 51.30, 48.10],
    'Baseline LSTM': [56.50, 53.20, 50.80],
    'MAGE+VAE':   [58.71, 55.82, 52.91],
    'MAGE+DAE':   [59.84, 57.01, 53.74],
})
print('Table III: MAGE Weighted F1 by Language')
print(table3.to_string(index=False))

## Table IV — Knowledge Distillation Results (Paper 3)

In [ ]:
table4 = pd.DataFrame({
    'Language':    ['Kinyarwanda', 'Swahili', 'Hausa', 'Igbo', 'Yoruba', 'Average'],
    'Teacher F1':  [68.91, 64.10, 79.30, 79.60, 74.20, 73.22],
    'Base F1':     [62.54, 60.81, 78.27, 78.63, 70.25, 70.10],
    'Mini F1':     [60.23, 61.88, 74.29, 75.20, 63.10, 66.94],
    'Comet F1':    [58.94, 55.30, 71.89, 73.82, 66.39, 65.28],
    'Params (M)':  [559.9,  559.9, 559.9, 559.9, 559.9, None],
    'Student (M)': [68.9,   68.9,  68.9,  68.9,  68.9,  None],
})
print('Table IV: AfroXLMR-Comet vs Teacher')
print(table4.to_string(index=False))

## Figure 2 — Efficiency vs Performance Tradeoff

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: inference time vs F1
models_d = ['XLM-R Large\n(Teacher)', 'XLM-R Base', 'XLM-R Mini', 'AfroXLMR\nComet']
times   = [293.9, 20.9, 15.3, 14.0]
f1s     = [73.22, 70.10, 66.94, 65.28]
sizes   = [2135, 449, 263, 263]  # MB

scatter = axes[0].scatter(times, f1s, s=[s/5 for s in sizes], c=['#D62728', '#FF7F0E', '#1F77B4', '#2CA02C'], zorder=5)
for i, m in enumerate(models_d):
    axes[0].annotate(m, (times[i], f1s[i]), textcoords='offset points', xytext=(5, 5), fontsize=8)
axes[0].set_xlabel('Inference time (ms)')
axes[0].set_ylabel('Average Weighted F1 (%)')
axes[0].set_title('Efficiency vs Performance')

# Right: parameter reduction
axes[1].bar(['Teacher', 'Comet Student'], [559.9, 68.9], color=['#D62728', '#2CA02C'])
axes[1].set_ylabel('Parameters (M)')
axes[1].set_title('Parameter Count: 87.7% Reduction')
for i, v in enumerate([559.9, 68.9]):
    axes[1].text(i, v + 5, f'{v}M', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../paper/figures/efficiency_tradeoff.pdf')
plt.show()